In [ ]:
UNI_RANDOM_SEED = 2024
DEVICE = 0

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

np.random.seed(UNI_RANDOM_SEED) 
torch.manual_seed(UNI_RANDOM_SEED)

torch.cuda.manual_seed(UNI_RANDOM_SEED)
torch.cuda.manual_seed_all(UNI_RANDOM_SEED)

torch.cuda.set_device(DEVICE)

import pdb
import pickle as pkl
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

def check_grad(input:np.ndarray):
    if input.shape.__len__() == 2:
        input = np.expand_dims(input, axis=1)
    
    # (1, 1)
    quad_1_mask = np.logical_and((input[:, :, 0] > 0),
                                 (input[:, :, 1] > 0))
    # (-1, 1)
    quad_2_mask = np.logical_and((input[:, :, 0] < 0),
                                 (input[:, :, 1] > 0))
    
    # (-1, -1)
    quad_3_mask = np.logical_and((input[:, :, 0] < 0),
                                 (input[:, :, 1] < 0))
    
    # (1, -1)
    quad_4_mask = np.logical_and((input[:, :, 0] > 0),
                                 (input[:, :, 1] < 0))
    
    quad_1_count = np.sum(quad_1_mask, axis=0)
    quad_2_count = np.sum(quad_2_mask, axis=0)
    quad_3_count = np.sum(quad_3_mask, axis=0)
    quad_4_count = np.sum(quad_4_mask, axis=0)
    
    return quad_1_count, quad_2_count, quad_3_count, quad_4_count

def draw_dis(input:np.ndarray, 
             title:str,
             color:str):
    # 生成一些示例数据
    data = input

    # 创建x轴坐标
    
    x = np.arange(input.shape[0])

    # 绘制曲线
    
    plt.figure(figsize=(16, 6))
    plt.ylim(0, 6500)
    plt.plot(x, data, color=color)

    # 填充曲线下面积，并指定颜色
    plt.fill_between(x, data, color=color, alpha=0.3)

    # 添加标题和标签
    plt.title(title)
    plt.xlabel('Index')
    plt.ylabel('Count')

    # 显示图形
    plt.show()

In [ ]:
grad = None
with open("./check_grad.pkl", "rb") as input:
    grad = pkl.load(input)
    
print(grad.__len__())

In [ ]:
grad[0]

In [ ]:
vertex_grad_numpy_flatten = []
translate_grad_numpy_flatten = []
theta_grad_numpy_flatten = []

for grad_list in grad:
    for grad_numpy in grad_list:
        vertex_grad_numpy_flatten.append(grad_numpy[0])
        translate_grad_numpy_flatten.append(grad_numpy[1])
        theta_grad_numpy_flatten.append(grad_numpy[2])
    
vertex_grad_numpy_flatten = np.stack(vertex_grad_numpy_flatten)
translate_grad_numpy_flatten = np.stack(translate_grad_numpy_flatten)
theta_grad_numpy_flatten = np.stack(theta_grad_numpy_flatten)

In [ ]:
vertex_quad = np.sign(vertex_grad_numpy_flatten[:, :, :2])
translate_quad = np.sign(translate_grad_numpy_flatten[:, :2])

vertex_quad = check_grad(vertex_quad)
translate_quad = check_grad(translate_quad)

In [ ]:
print(translate_quad)

In [ ]:
draw_dis(vertex_quad[0], 
         "Quad 1",
         "red")

draw_dis(vertex_quad[1], 
         "Quad 2",
         "blue")

draw_dis(vertex_quad[2], 
         "Quad 3",
         "green")

draw_dis(vertex_quad[3], 
         "Quad 4",
         "skyblue")